# ACT training on the Track 4 + PID LeRobot dataset (Colab)

**What this notebook does.** Trains an ACT policy (`lerobot==0.4.4`, `policy.type=act`, action chunk length 100) on the
LeRobot v3.0 dataset written by `host_software/ml_jetson_vla/data_processing/convert_to_lerobot.py`, then evaluates it on
held-out sessions. Training runs through lerobot's own `train()` entry point (launched as its CLI, see cell 6); there is no bespoke loop.

- **Trains ONLY on the `train` split** of `session_split.json`. The unit is a whole session, never a frame.
- **Eval is by session.** The `eval` sessions are never passed to training. They are used only in the eval cell
  (action MSE in theta-degrees, per episode and overall).
- **The Jetson is not used for training.** It is the inference/deployment target only.

**Placeholders to set in cell 1:** `DRIVE_BOOTSTRAP_DIR`, `DATASET_ZIP`, `DRIVE_RUN_DIR`.

**Progress and resume.** Each training step is appended to a JSONL file on Drive. Checkpoints go to Drive every `SAVE_FREQ`
steps. Re-running cell 6 resumes from the last checkpoint. The eval cell writes its per-batch progress to Drive too.

**Known open issue (not handled here).** Frames with `touch_stale == 1` (spec risk #3, about 34% of the dry-run frames) are
included in training. This notebook does not mask them; that is a decision for the user.

**Status.** This notebook has NOT been run. It was checked locally only (JSON validity, compile, lerobot name existence,
and the split logic on fake episode tables).

In [ ]:
# (a) GPU check. Placeholders are set here too.
import subprocess

# ---- PLACEHOLDERS: set these ----
DRIVE_BOOTSTRAP_DIR = "/content/drive/MyDrive/vri2026_track4_bootstrap"  # PLACEHOLDER: Drive folder with the zip, split, manifest
DATASET_ZIP = "PLACEHOLDER_lerobot_dataset.zip"                         # PLACEHOLDER: dataset zip name in that folder
DRIVE_RUN_DIR = "/content/drive/MyDrive/vri2026_track4_bootstrap/act_runs/run_001"  # PLACEHOLDER: where outputs go
# ---------------------------------
SESSION_SPLIT_NAME = "session_split.json"
SESSION_MANIFEST_NAME = "session_manifest.json"
LOCAL_WORK = "/content/work"
REPO_ID = "vri2026/combined_jetson_track4_and_pid"  # the converter's default repo_id; must match the dataset info if it changed
SEED = 20261006  # same seed as session_split.json

out = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("nvidia-smi:", out)
GPU_NAME, COMPUTE_CAP_STR, GPU_MEM = [s.strip() for s in out.splitlines()[0].split(",")]
ALLOWED_GPU_CLASSES = ("A100", "L4", "T4")
assert any(c in GPU_NAME for c in ALLOWED_GPU_CLASSES), f"GPU {GPU_NAME!r} is not one of {ALLOWED_GPU_CLASSES}"
COMPUTE_CAP = tuple(int(x) for x in COMPUTE_CAP_STR.split("."))
# bf16 needs Ampere (sm_80+). T4 is sm_75 and gets fp16. Used as ACCELERATE_MIXED_PRECISION (autocast, fp32 weights).
MIXED_PRECISION = "bf16" if COMPUTE_CAP >= (8, 0) else "fp16"
print(f"device: {GPU_NAME} (compute capability {COMPUTE_CAP_STR}) -> mixed precision: {MIXED_PRECISION}")

In [ ]:
# (b) Pin versions to the Jetson arm2-lerobot image's stack.
# torch 2.7.0 / transformers 4.57.6 are the versions recorded for the image in LARGE_VLA_RESEARCH_SPIKE.md (2026-10-06).
# lerobot==0.4.4 is the pinned lerobot. Cell (b) installs packages, so restart the runtime first
# if torch was already imported in this session, then run from cell (a).
import subprocess, sys

PIN_LEROBOT = "0.4.4"
PIN_TORCH = "2.7.0"
PIN_TORCHVISION = "0.22.0"   # torchvision 0.22.x pairs with torch 2.7.x
PIN_TORCHCODEC = "0.4.*"     # torchcodec 0.4 pairs with torch 2.7 (verify with the decode preflight in cell (d))
PIN_TRANSFORMERS = "4.57.6"  # lerobot's transformers<5 requirement; the Jetson image resolved 4.57.6

def _pip(*args: str, extra: list = None) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    if extra:
        cmd += extra
    subprocess.run(cmd, check=True)

_pip(f"torch=={PIN_TORCH}", f"torchvision=={PIN_TORCHVISION}",
     extra=["--index-url", "https://download.pytorch.org/whl/cu126"])
_pip(f"torchcodec=={PIN_TORCHCODEC}", f"transformers=={PIN_TRANSFORMERS}")
_pip(f"lerobot=={PIN_LEROBOT}")

VERSION_CHECK = (
    "import importlib.metadata as m, torch, torchvision; "
    "print('torch', torch.__version__); print('torchvision', torchvision.__version__); "
    "print('cuda_available', torch.cuda.is_available()); print('lerobot', m.version('lerobot')); "
    "print('transformers', m.version('transformers')); print('torchcodec', m.version('torchcodec'))"
)
versions_txt = subprocess.run([sys.executable, "-c", VERSION_CHECK], capture_output=True, text=True, check=True).stdout
print(versions_txt)
VERSIONS = dict(line.split(" ", 1) for line in versions_txt.strip().splitlines())
assert VERSIONS["torch"].split("+")[0] == PIN_TORCH, f"torch {VERSIONS['torch']} != {PIN_TORCH}"
assert VERSIONS["lerobot"] == PIN_LEROBOT, f"lerobot {VERSIONS['lerobot']} != {PIN_LEROBOT}"
assert VERSIONS["cuda_available"] == "True", "CUDA not visible to the freshly installed torch"

In [ ]:
# (c) Mount Drive, copy the dataset zip, split and manifest; unzip to LOCAL disk (never Drive).
import glob, hashlib, json, os, shutil, zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(LOCAL_WORK, exist_ok=True)

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

src_zip = os.path.join(DRIVE_BOOTSTRAP_DIR, DATASET_ZIP)
assert os.path.exists(src_zip), f"missing dataset zip: {src_zip}"
local_zip = os.path.join(LOCAL_WORK, DATASET_ZIP)
shutil.copy(src_zip, local_zip)
DATASET_ZIP_SHA256 = sha256_file(local_zip)
print("dataset zip sha256:", DATASET_ZIP_SHA256)

EXTRACT_DIR = os.path.join(LOCAL_WORK, "dataset")
shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(EXTRACT_DIR)
info_paths = glob.glob(os.path.join(EXTRACT_DIR, "**", "meta", "info.json"), recursive=True)
assert len(info_paths) == 1, f"expected exactly one meta/info.json in the zip, found {info_paths}"
DATASET_ROOT = os.path.dirname(os.path.dirname(info_paths[0]))
with open(info_paths[0]) as f:
    info = json.load(f)
assert info["codebase_version"] == "v3.0", info["codebase_version"]
print("dataset root:", DATASET_ROOT, "| episodes:", info["total_episodes"], "| frames:", info["total_frames"])

SPLIT_PATH = os.path.join(LOCAL_WORK, SESSION_SPLIT_NAME)
MANIFEST_PATH = os.path.join(LOCAL_WORK, SESSION_MANIFEST_NAME)
shutil.copy(os.path.join(DRIVE_BOOTSTRAP_DIR, SESSION_SPLIT_NAME), SPLIT_PATH)
shutil.copy(os.path.join(DRIVE_BOOTSTRAP_DIR, SESSION_MANIFEST_NAME), MANIFEST_PATH)
SPLIT_SHA256 = sha256_file(SPLIT_PATH)
MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)  # Drive write: run folder exists before anything long starts

In [ ]:
# (d) Load split, map sessions to episodes via the converter's sidecar, check regimes, assert disjointness, print counts.
import glob, json, os
import pandas as pd

with open(SPLIT_PATH) as f:
    SPLIT = json.load(f)
with open(MANIFEST_PATH) as f:
    MANIFEST = json.load(f)

# --- PURE SPLIT LOGIC (unit-checked locally) ---
TRACK4_PREFIX = "session_jetson_track4_"
TRACK4_REGIME = "jetson_track4_rl"
PID_REGIME = "pid_webcam"

def check_episode_regimes(order: list, episode_regime: dict) -> None:
    for i, (name, expected) in enumerate(order):
        got = episode_regime.get(i)
        assert got == expected, f"episode {i} regime {got!r} != expected {expected!r} (session {name})"

def split_to_episodes(split: dict, session_to_episode: dict):
    train_eps = sorted(session_to_episode[s] for s in split["train"])
    eval_eps = sorted(session_to_episode[s] for s in split["eval"])
    assert train_eps and eval_eps, "train and eval must both be non-empty"
    assert len(set(train_eps)) == len(train_eps) and len(set(eval_eps)) == len(eval_eps)
    assert not (set(train_eps) & set(eval_eps)), f"episode overlap: {set(train_eps) & set(eval_eps)}"
    return train_eps, eval_eps

def per_regime_frame_counts(episodes: list, episode_length: dict, episode_regime: dict) -> dict:
    out = {}
    for e in episodes:
        r = episode_regime[e]
        out[r] = out.get(r, 0) + int(episode_length[e])
    return out
# --- END PURE SPLIT LOGIC ---

def load_episode_tables(root: str):
    data_files = sorted(glob.glob(os.path.join(root, "data", "**", "*.parquet"), recursive=True))
    df = pd.concat([pd.read_parquet(p, columns=["episode_index", "regime"]) for p in data_files])
    assert (df.groupby("episode_index")["regime"].nunique() == 1).all(), "an episode mixes regimes"
    episode_regime = {int(k): v for k, v in df.groupby("episode_index")["regime"].first().items()}
    meta_files = sorted(glob.glob(os.path.join(root, "meta", "episodes", "**", "*.parquet"), recursive=True))
    meta = pd.concat([pd.read_parquet(p, columns=["episode_index", "length"]) for p in meta_files])
    episode_length = {int(k): int(v) for k, v in meta.set_index("episode_index")["length"].items()}
    return episode_regime, episode_length

EPISODE_REGIME, EPISODE_LENGTH = load_episode_tables(DATASET_ROOT)
N_EPISODES = len(EPISODE_LENGTH)

# The converter writes the session-to-episode mapping; it is read here, not reconstructed from job order.
SIDECAR_PATH = os.path.join(DATASET_ROOT, "meta", "session_episodes.json")
assert os.path.exists(SIDECAR_PATH), "dataset has no session_episodes.json -- rebuild it with the current converter"
SIDECAR = json.load(open(SIDECAR_PATH))["episodes"]
SESSION_TO_EPISODE = {r["session_name"]: int(r["episode_index"]) for r in SIDECAR}
assert set(SPLIT["train"]) | set(SPLIT["eval"]) <= set(SESSION_TO_EPISODE), "split sessions missing from sidecar"
ORDER = [(r["session_name"], r["regime"]) for r in sorted(SIDECAR, key=lambda r: r["episode_index"])]
assert len(ORDER) == N_EPISODES, f"sidecar has {len(ORDER)} episodes, dataset has {N_EPISODES}"
check_episode_regimes(ORDER, EPISODE_REGIME)
TRAIN_EPISODES, EVAL_EPISODES = split_to_episodes(SPLIT, SESSION_TO_EPISODE)
EPISODE_TO_SESSION = {v: k for k, v in SESSION_TO_EPISODE.items()}
print(f"episodes in dataset: {N_EPISODES}; train episodes: {len(TRAIN_EPISODES)}; eval episodes: {len(EVAL_EPISODES)}")
print("TRAIN sessions:", [EPISODE_TO_SESSION[e] for e in TRAIN_EPISODES])
print("EVAL  sessions:", [EPISODE_TO_SESSION[e] for e in EVAL_EPISODES])
print("train per-regime frames:", per_regime_frame_counts(TRAIN_EPISODES, EPISODE_LENGTH, EPISODE_REGIME))
print("eval  per-regime frames:", per_regime_frame_counts(EVAL_EPISODES, EPISODE_LENGTH, EPISODE_REGIME))

# Decode preflight: AV1 video (info.json says codec av1) must decode with the installed backend.
from lerobot.datasets.lerobot_dataset import LeRobotDataset
probe = LeRobotDataset(REPO_ID, root=DATASET_ROOT, episodes=[TRAIN_EPISODES[0]])
sample = probe[0]
print("decode preflight: observation.image shape", tuple(sample["observation.image"].shape),
      "| state", sample["observation.state"].tolist(), "| action", sample["action"].tolist())
assert tuple(sample["observation.image"].shape)[0] == 3


In [ ]:
# (e) ACT config. chunk_size=100 is lerobot's ACTConfig default (chunk_size=100, n_action_steps=100 in
# configuration_act.py); it is not a new value. Input features come from the dataset metadata, not hand-written.
from lerobot.configs.types import FeatureType
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.act.configuration_act import ACTConfig

feats = dataset_to_policy_features(probe.meta.features)
input_features = {k: v for k, v in feats.items() if v.type is not FeatureType.ACTION}
output_features = {k: v for k, v in feats.items() if v.type is FeatureType.ACTION}
assert input_features["observation.state"].shape == (2,), input_features["observation.state"]
assert output_features["action"].shape == (3,), output_features["action"]
print("input features:", {k: (v.type.value, v.shape) for k, v in input_features.items()})
print("output features:", {k: (v.type.value, v.shape) for k, v in output_features.items()})

ACT_CHUNK = ACTConfig.__dataclass_fields__["chunk_size"].default
assert ACT_CHUNK == 100
ACT_DEFAULTS = {n: ACTConfig.__dataclass_fields__[n].default for n in ("chunk_size", "n_action_steps", "use_vae", "latent_dim", "kl_weight", "vision_backbone")}
print("ACT class defaults:", ACT_DEFAULTS)

# Training hyperparameters that are NOT defaults of anything in the repo are marked as placeholders.
STEPS = 100_000          # lerobot TrainPipelineConfig default; PLACEHOLDER to set for the real run
SAVE_FREQ = 2_000        # PLACEHOLDER: checkpoint every N steps (N chosen here; CLAUDE.md >30-min checkpoint rule)
BATCH_SIZE = 8           # lerobot TrainPipelineConfig default
NUM_WORKERS = 2
OUTPUT_DIR = os.path.join(DRIVE_RUN_DIR, "train_output")  # checkpoints live on Drive
STEP_LOG_PATH = os.path.join(DRIVE_RUN_DIR, "train_steps.jsonl")
STDOUT_LOG_PATH = os.path.join(DRIVE_RUN_DIR, "train_stdout.log")
print("run dir:", DRIVE_RUN_DIR, "| steps:", STEPS, "| save_freq:", SAVE_FREQ, "| batch:", BATCH_SIZE)

In [ ]:
# (f) Training through lerobot's own train() entry point. Launched as a subprocess because lerobot's
# train() is CLI-parsed (parser.wrap reads sys.argv) and calls init_logging(), which clears root handlers.
# The launcher below only adds a logging handler after init_logging() runs, so each step's MetricsTracker
# (the object lerobot passes to logging.info) is written to Drive as JSONL. Nothing is parsed from text.
import json, logging, os, subprocess, sys

LAUNCHER_PATH = os.path.join(LOCAL_WORK, "act_launch.py")
LAUNCHER_SRC = '''
import json, logging, os, time
import lerobot.scripts.lerobot_train as lt
from lerobot.utils.logging_utils import MetricsTracker

class _StepJsonlHandler(logging.Handler):
    def __init__(self, path):
        super().__init__()
        self.path = path
    def emit(self, record):
        msg = record.msg
        if not isinstance(msg, MetricsTracker):
            return
        row = {
            "step": int(msg.steps),
            "loss": float(msg.metrics["loss"].val),
            "grad_norm": float(msg.metrics["grad_norm"].val),
            "lr": float(msg.metrics["lr"].val),
            "wall_time": time.time(),
        }
        with open(self.path, "a") as f:
            f.write(json.dumps(row) + "\\n")

_orig_init_logging = lt.init_logging
def _init_logging_then_hook(*args, **kwargs):
    _orig_init_logging(*args, **kwargs)
    logging.getLogger().addHandler(_StepJsonlHandler(os.environ["ACT_STEP_LOG"]))
lt.init_logging = _init_logging_then_hook

lt.main()
'''
with open(LAUNCHER_PATH, "w") as f:
    f.write(LAUNCHER_SRC)

last_cfg = os.path.join(OUTPUT_DIR, "checkpoints", "last", "pretrained_model", "train_config.json")
resume_args = []
if os.path.exists(last_cfg):
    resume_args = ["--resume=true", f"--config_path={last_cfg}"]
    print("resuming from", last_cfg)
elif os.path.isdir(OUTPUT_DIR):
    raise RuntimeError(f"{OUTPUT_DIR} exists without a last checkpoint; lerobot refuses to overwrite it. Move it aside.")

train_args = [
    "--policy.type=act",
    f"--policy.chunk_size={ACT_CHUNK}",
    f"--policy.n_action_steps={ACT_CHUNK}",
    "--policy.device=cuda",
    "--policy.push_to_hub=false",
    f"--dataset.repo_id={REPO_ID}",
    f"--dataset.root={DATASET_ROOT}",
    f"--dataset.episodes={json.dumps(TRAIN_EPISODES)}",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=act_track4_pid",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    f"--num_workers={NUM_WORKERS}",
    f"--save_freq={SAVE_FREQ}",
    "--log_freq=1",
    f"--seed={SEED}",
    "--eval_freq=0",
    "--wandb.enable=false",
] + resume_args

env = dict(os.environ, ACCELERATE_MIXED_PRECISION=MIXED_PRECISION, ACT_STEP_LOG=STEP_LOG_PATH)
with open(STDOUT_LOG_PATH, "a") as log:
    proc = subprocess.run([sys.executable, LAUNCHER_PATH, *train_args], env=env,
                          stdout=log, stderr=subprocess.STDOUT)
if proc.returncode != 0:
    raise RuntimeError(f"lerobot train exited {proc.returncode}; see {STDOUT_LOG_PATH} (checkpoints so far are on Drive)")
print("training finished; checkpoints under", os.path.join(OUTPUT_DIR, "checkpoints"))

In [ ]:
# (g) Summarise the per-step JSONL into one JSON array on Drive, and check grad norms and loss are finite.
import json, math, os

rows = [json.loads(line) for line in open(STEP_LOG_PATH)] if os.path.exists(STEP_LOG_PATH) else []
assert rows, "no step rows yet"
bad = [r for r in rows if not (math.isfinite(r["loss"]) and math.isfinite(r["grad_norm"]))]
assert not bad, f"non-finite loss/grad_norm at {len(bad)} steps, first: {bad[0]}"
GRAD_LOSS_JSON = os.path.join(DRIVE_RUN_DIR, "grad_loss_log.json")
with open(GRAD_LOSS_JSON, "w") as f:
    json.dump(rows, f)
LAST_STEP = rows[-1]["step"]
print(f"{len(rows)} logged steps; last step {LAST_STEP}; last loss {rows[-1]['loss']:.4f}; "
      f"last grad_norm {rows[-1]['grad_norm']:.4f}; wrote {GRAD_LOSS_JSON}")

In [ ]:
# (h) Eval on the held-out EVAL episodes only. Action MSE in theta-degrees, per episode and overall.
# Uses the saved pre/post-processors (normalisation lives there in lerobot 0.4.4) and predict_action_chunk()
# on each frame; the metric compares chunk step 0 (postprocessed) with the recorded action at that frame.
# fp32 inference, no autocast (the spike found bf16/autocast issues on the ACT inference path).
import json, os
from collections import defaultdict
import torch
from torch.utils.data import DataLoader
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors

PRETRAINED_DIR = os.path.join(OUTPUT_DIR, "checkpoints", "last", "pretrained_model")
EVAL_BATCH = 32
EVAL_PROGRESS = os.path.join(DRIVE_RUN_DIR, "eval_progress.jsonl")
EVAL_RESULTS = os.path.join(DRIVE_RUN_DIR, "eval_results.json")

policy = ACTPolicy.from_pretrained(PRETRAINED_DIR).to("cuda").eval()
preprocessor, postprocessor = make_pre_post_processors(policy_cfg=policy.config, pretrained_path=PRETRAINED_DIR)

eval_ds = LeRobotDataset(REPO_ID, root=DATASET_ROOT, episodes=EVAL_EPISODES)
assert sorted(set(int(e) for e in eval_ds.hf_dataset["episode_index"])) == EVAL_EPISODES
loader = DataLoader(eval_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=NUM_WORKERS)

sq_sum = defaultdict(lambda: torch.zeros(3, dtype=torch.float64))
n_frames = defaultdict(int)
frames_done = 0
if os.path.exists(EVAL_PROGRESS):
    os.remove(EVAL_PROGRESS)
with torch.no_grad():
    for batch in loader:
        gt = batch["action"].detach().cpu().double()           # degrees, recorded
        ep_ids = [int(e) for e in batch["episode_index"]]
        pre = preprocessor(dict(batch))
        chunk = policy.predict_action_chunk(pre)               # (B, chunk, 3), normalised space
        pred = postprocessor(chunk[:, 0]).detach().cpu().double()  # (B, 3), degrees
        err2 = (pred - gt) ** 2
        for i, e in enumerate(ep_ids):
            sq_sum[e] += err2[i]
            n_frames[e] += 1
        frames_done += len(ep_ids)
        with open(EVAL_PROGRESS, "a") as f:
            f.write(json.dumps({"frames_done": frames_done}) + "\n")

per_episode = {}
total_sq = torch.zeros(3, dtype=torch.float64)
total_n = 0
for e in sorted(n_frames):
    mse_j = (sq_sum[e] / n_frames[e]).tolist()
    per_episode[EPISODE_TO_SESSION[e]] = {
        "episode_index": e, "frames": n_frames[e],
        "mse_deg2_per_joint": mse_j, "mse_deg2": sum(mse_j) / 3.0,
    }
    total_sq += sq_sum[e]
    total_n += n_frames[e]
overall_joint = (total_sq / total_n).tolist()
EVAL_SUMMARY = {"frames": total_n, "mse_deg2_per_joint": overall_joint, "mse_deg2": sum(overall_joint) / 3.0}
with open(EVAL_RESULTS, "w") as f:
    json.dump({"per_episode": per_episode, "overall": EVAL_SUMMARY}, f, indent=2)
for s, v in per_episode.items():
    print(f"{s}: frames={v['frames']} MSE(deg^2)={v['mse_deg2']:.3f} per-joint={[round(x, 3) for x in v['mse_deg2_per_joint']]}")
print("OVERALL (frame-weighted):", EVAL_SUMMARY)

In [ ]:
# (i) Final artefacts on Drive: copy of the trained checkpoint, and run_config.json.
import json, os, shutil
from importlib import metadata

final_ckpt_dir = os.path.join(DRIVE_RUN_DIR, "final_pretrained_model")
shutil.copytree(PRETRAINED_DIR, final_ckpt_dir, dirs_exist_ok=True)

with open(os.path.join(PRETRAINED_DIR, "train_config.json")) as f:
    saved_train_cfg = json.load(f)

run_config = {
    "dataset_zip": DATASET_ZIP,
    "dataset_zip_sha256": DATASET_ZIP_SHA256,
    "split_file": SESSION_SPLIT_NAME,
    "split_file_sha256": SPLIT_SHA256,
    "manifest_file": SESSION_MANIFEST_NAME,
    "manifest_sha256": MANIFEST_SHA256,
    "repo_id": REPO_ID,
    "gpu": GPU_NAME,
    "versions": VERSIONS,
    "seed": SEED,
    "steps_requested": STEPS,
    "last_logged_step": LAST_STEP,
    "chunk_length": ACT_CHUNK,
    "n_action_steps": ACT_CHUNK,
    "policy_defaults_used": ACT_DEFAULTS,
    "precision": MIXED_PRECISION,
    "batch_size": BATCH_SIZE,
    "save_freq": SAVE_FREQ,
    "train_episodes": TRAIN_EPISODES,
    "eval_episodes": EVAL_EPISODES,
    "train_sessions": [EPISODE_TO_SESSION[e] for e in TRAIN_EPISODES],
    "eval_sessions": [EPISODE_TO_SESSION[e] for e in EVAL_EPISODES],
    "eval_summary": EVAL_SUMMARY,
    "train_config_saved_by_lerobot": saved_train_cfg,
}
RUN_CONFIG_PATH = os.path.join(DRIVE_RUN_DIR, "run_config.json")
with open(RUN_CONFIG_PATH, "w") as f:
    json.dump(run_config, f, indent=2, default=str)
print("wrote", RUN_CONFIG_PATH, "and checkpoint", final_ckpt_dir)